In [ ]:
# ==============================================================================
# CELL 1: DEPENDENCIES & FOUNDATION ENVIRONMENT SETUP
# ==============================================================================
import os
import random
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ SOTA Foundational environment anchored on device: {device}")

In [ ]:
# ==============================================================================
# CELL 2: INGEST DATA & INITIALIZE SOTA EMBEDDING EXTRACTOR
# ==============================================================================
print("Ingesting molecular dataset profiles...")
# train_df         = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TRAIN.csv") # 4140 molecules that passed the single concentration test?
# test_df          = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TEST_BLINDED.csv") # 513 (63 + analog) final filtered molecules with high pEC50 AND counter assay success?
train_df = pd.read_csv("../data/MOE_TRAIN.txt")
test_df = pd.read_csv("../data/MOE_TEST.txt")

target_cols = ['pEC50', 'Emax_estimate(log2FCvs.baseline)', 'Emax.vs.pos.ctrl_estimate(dimensionless)']
train_df = train_df.dropna(subset=target_cols + ['SMILES.smiles']).reset_index(drop=True)

# Define our SOTA Foundation Model ID (ChemBERTa-2 with Multi-Task pre-training)
SOTA_MODEL_ID = "deepchem/ChemBERTa-77M-MTR"
print(f"Downloading SOTA Tokenizer and Weights for: {SOTA_MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(SOTA_MODEL_ID)
foundation_model = AutoModel.from_pretrained(SOTA_MODEL_ID).to(device)
foundation_model.eval()

def extract_sota_embeddings(smiles_list, model, tokenizer, batch_size=32):
    """Passes SMILES sequences through ChemBERTa-2 to harvest 600-dimensional pooled hidden states."""
    embeddings = []
    for i in tqdm(range(0, len(smiles_list), batch_size), desc="Harvesting SOTA Latent Space"):
        batch_smiles = smiles_list[i:i+batch_size].tolist()
        
        # Tokenize SMILES strings directly into attention masks
        encoded = tokenizer(batch_smiles, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model(**encoded)
            # Use Mean Pooling over the token sequence to derive the global molecular vector
            attention_mask = encoded['attention_mask'].unsqueeze(-1)
            token_embeddings = outputs.last_hidden_state
            pooled = torch.sum(token_embeddings * attention_mask, dim=1) / torch.clamp(attention_mask.sum(dim=1), min=1e-9)
            embeddings.append(pooled.cpu().numpy())
            
    return np.vstack(embeddings)

print("\nExtracting baseline SOTA representations...")
X_train_sota = extract_sota_embeddings(train_df['SMILES.smiles'], foundation_model, tokenizer)
X_test_sota = extract_sota_embeddings(test_df['SMILES.smiles'], foundation_model, tokenizer)

# Clean and calculate scaffolds for grouping
def get_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return "invalid"
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    except: return "invalid"

train_df['scaffold'] = [get_scaffold(s) for s in train_df['SMILES.smiles']]
train_df['is_active'] = (train_df['pEC50'] >= 6.5) & (train_df['Emax_estimate(log2FCvs.baseline)'] >= 2.0)
y_train_pec50 = train_df['pEC50'].values

print(f"\nSOTA Hidden Dimension Matrix Shape: {X_train_sota.shape}")

In [ ]:
# ==============================================================================
# CELL 3: TRIPLET DATA LOADER (ACTIVITY CLIFF MINER)
# ==============================================================================
class SOTATripletDataset(Dataset):
    """
    Pairs SOTA foundation embeddings into functional triplets:
    - Anchor: Active Agonist Embeddings
    - Positive: Distant Active Agonist Embeddings
    - Negative: Structural Scaffold Match with Inactive Profile (The Cliff)
    """
    def __init__(self, df, sota_matrix):
        self.df = df
        self.features = sota_matrix
        self.scaffolds = df['scaffold'].values
        self.actives_idx = df[df['is_active'] == True].index.values
        self.inactives_idx = df[df['is_active'] == False].index.values
        self.triplets = self._mine_cliffs()

    def _mine_cliffs(self):
        triplets = []
        scaffold_groups = self.df.groupby('scaffold').groups
        
        for idx in self.actives_idx:
            anchor_scaffold = self.scaffolds[idx]
            scaffold_mates = scaffold_groups[anchor_scaffold]
            negatives = [m for m in scaffold_mates if m in self.inactives_idx]
            
            if len(negatives) == 0: negatives = self.inactives_idx
            positives = [p for p in self.actives_idx if self.scaffolds[p] != anchor_scaffold]
            if len(positives) == 0: positives = self.actives_idx
                
            for neg in negatives[:3]:
                pos = random.choice(positives)
                triplets.append((idx, pos, neg))
        return triplets

    def __len__(self): return len(self.triplets)

    def __getitem__(self, idx):
        a_idx, p_idx, n_idx = self.triplets[idx]
        return (
            torch.tensor(self.features[a_idx], dtype=torch.float32),
            torch.tensor(self.features[p_idx], dtype=torch.float32),
            torch.tensor(self.features[n_idx], dtype=torch.float32)
        )

In [ ]:
# ==============================================================================
# CELL 4: CONTRASTIVE FINE-TUNING NETWORK (THE CLIFF SEPARATOR)
# ==============================================================================
class SOTAContrastiveAdapter(nn.Module):
    """
    Fine-tunes the pre-trained SOTA space. It maps the 600-dimensional ChemBERTa space
    into an optimized hypersphere configured to prioritize Helix-12 behavior.
    """
    def __init__(self, input_dim=600, output_dim=256):
        super().__init__()
        self.adapter = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, output_dim)
        )
        
    def forward(self, x):
        return F.normalize(self.adapter(x), p=2, dim=1)

In [ ]:
# ==============================================================================
# CELL 5: REPRESENTATIONAL ALIGNMENT TRAINING LOOP
# ==============================================================================
triplet_dataset = SOTATripletDataset(train_df, X_train_sota)
triplet_loader = DataLoader(triplet_dataset, batch_size=64, shuffle=True, drop_last=True)

adapter = SOTAContrastiveAdapter(input_dim=X_train_sota.shape[1], output_dim=256).to(device)
optimizer = torch.optim.AdamW(adapter.parameters(), lr=1e-4, weight_decay=1e-3)
criterion = nn.TripletMarginLoss(margin=1.2, p=2) # Higher margin aggressively isolates cliffs

print("Fine-tuning foundational representation landscape...")
adapter.train()
for epoch in range(20):
    epoch_loss = 0.0
    for anchor, positive, negative in triplet_loader:
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
        
        optimizer.zero_grad()
        
        # Transform the frozen foundation states into our fine-tuned space
        a_space = adapter(anchor)
        p_space = adapter(positive)
        n_space = adapter(negative)
        
        loss = criterion(a_space, p_space, n_space)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/20 | Foundational Contrastive Loss: {epoch_loss/len(triplet_loader):.4f}")

print("✅ Foundational space successfully reshaped around target activity profiles.")

In [ ]:
# ==============================================================================
# CELL 6: SCAFFOLD EVALUATION VALIDATION PASS
# ==============================================================================
adapter.eval()
with torch.no_grad():
    X_train_tensor = torch.tensor(X_train_sota, dtype=torch.float32).to(device)
    X_test_tensor = torch.tensor(X_test_sota, dtype=torch.float32).to(device)
    
    # Extract the fine-tuned biophysical vector layers
    X_train_tuned = adapter(X_train_tensor).cpu().numpy()
    X_test_tuned = adapter(X_test_tensor).cpu().numpy()

# Concat original SOTA structural features with our Activity-Cliff fine-tuned coordinates
X_train_final = np.hstack([X_train_sota, X_train_tuned])
X_test_final = np.hstack([X_test_sota, X_test_tuned])

# Cross Validation using GroupKFold to maintain strict scaffold isolation
scaffold_to_group = {s: i for i, s in enumerate(train_df["scaffold"].unique())}
groups = train_df["scaffold"].map(scaffold_to_group).values
gkf = GroupKFold(n_splits=5)

def calculate_rae(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true - np.mean(y_true)))

fold_raes = []
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_final, y_train_pec50, groups), 1):
    X_tr, X_val = X_train_final[train_idx], X_train_final[val_idx]
    y_tr, y_val = y_train_pec50[train_idx], y_train_pec50[val_idx]
    
    regressor = LGBMRegressor(n_estimators=1500, learning_rate=0.03, max_depth=6, random_state=42, verbose=-1)
    regressor.fit(X_tr, y_tr)
    
    preds = regressor.predict(X_val)
    fold_raes.append(calculate_rae(y_val, preds))

print(f"\n➔ SOTA Contrastive-Finetuned CV RAE: {np.mean(fold_raes):.4f}")

In [ ]:
# ==============================================================================
# CELL 7: FINAL COMPETITION SUBMISSION DEPLOYMENT
# ==============================================================================
print("\nTraining final production regressor across all fine-tuned matrices...")
final_regressor = LGBMRegressor(n_estimators=1500, learning_rate=0.03, max_depth=6, random_state=42, verbose=-1)
final_regressor.fit(X_train_final, y_train_pec50)
final_test_preds = final_regressor.predict(X_test_final)

submission_df = pd.DataFrame({
    'MoleculeName': test_df['MoleculeName'],
    'pEC50': final_test_preds
})

submission_df.to_csv("submission_day12_sota_contrastive.csv", index=False)
print("✅ SOTA fine-tuned pipeline successfully deployed inside 'submission_day12_sota_contrastive.csv'.")